

 Тестирование проводилось на файле 5580_СВОП_для_сравнения (письмо от 19 мая)

1.   В файле спотовые курсы подгружаются CNYRUB_TOM, а не просто CNYRUB
2.   Расчет СС ведется через фиксрованный курс 11,8368








> Все внешние данные подключены к DataArchive и корректно парсятся


> **Pros**


Данные все подгружаются хорошо (спот курсы, своп-поинты и ставки соответсвуют данным в DataArchive), если на эту дату нет данных, то выдает ошибку (можно сделать, чтобы подгружались данные за предыдущий день и в отдельном столбце будет пометка, если за предыдущий)






> **Cons**



1) Даты из экселя плохо преобразуются через pd.to_datetime 01.08.2024 спокойно превращается в 08.01.2024 - решено через фиксированный формат даты (условие dayfirst = True и даты в файле Excel указаны в формате кратких дат)



2) Ставки. Если не прописывать конкретную ставку тянется самая первая, с учетом источника. Либо прописываем для каждой валюты свою ставку в изначальных условиях через словарь, либо определять хардами.

как вариант (надо обсудить и согласовать по какой валюте какая ставка будет тянуться, сделаем словарик и пропишем в sql запросе тип ставки) - так будет понятно что за ставка

Рубли - RUB_RUB/RUONIA (если есть данные)
Юани - CNY_CNY, SHIBOR
Доллары - SOFR
Евро - ESTR

Для удобства можно сделать столбец с источниками данных по крайней мере по ставкам

**UPD**: сделала словарь и ставку для каждой валюты, теперь ставка тянется в соответсвии с ним функцией перед парсингом из DataArchive


3) При правильных ставках своп-поинты не всегда совпадают с Банком (очень узкий диаппазон по типу 0.001776 - 0.001791, в файле эксель 0.0018 - 0.0057, а своп-поинты по сделке 0.00319 - теоретически не сошлось бы через ставки)

сейчас своп-поинты считаются следующим образом: Спот мин/макс * ((1+ интерполированная ставка по внутренней валюте/100)/(1+интерполированаая ставка по внешней валюте/100)) ** Срок/365 - Спот мин/макс

В коде есть часть с другой формулой расчета св


4) Расхождение своп в % не обсчитывается когда все "в рынке", вместо нет данных будет "-"


5) Справедливая стоимость не совпадает при идентичных исходных данных и в целом рассчитывается странно, еще раз формулу согласовать надо

Текущая формула: Сумма покупки (2 часть сделки) * дискаунт фактор 1 * Курс на дату оценки_1 - сумма продажи (2 часть сделки) * дискаунт фактор 2 * Курс на дату оценки_2




6) При загрузке реестра Банка надо обработать файл, так как сумма продажи/покупки есть как в первой, так и во второй части сделки, а столбцы называются одинаково, можно было бы прописать конкретные столбцы, но в реестре может не быть столбца режима торгов, биржевая/внебиржевая (сравнила несколько реестров, заполнены немного по-разному) и из-за этого номера собьются

Либо все нужные столбцы переимновывать отдельно перед началом расчета файла, соответсвенно закрепить имена столбцов для всех операций внутри кода и сылаться исключительно на них

Но даже в таком случае изнчально надо проверить корректность самого реестра (как полностью автоматизировать этот момент еще подумаю, каждый раз проверять тоже неудобюно)

# Libraries

In [ ]:
!pip install QuantLib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 23.1 MB/s eta 0:00:00


In [ ]:
import QuantLib as ql
import numpy as np
import pandas as pd
import scipy as sc
import openpyxl
import codecs
import copy
import os
import time
from datetime import datetime
from contextlib import closing
import sqlite3
from scipy.interpolate import interp1d


In [ ]:
import warnings
warnings.filterwarnings('ignore')

# **Data parsing**

In [ ]:
# Подключение Google Drive
from google.colab import drive # Тут необходимо подлючится к диску через Google аккаунт
drive.mount('/content/drive', force_remount=False) # В одной сессии хватит подключится один раз, дальше подлючение не требуется (только при перезапуске сессии)

In [ ]:
# Настройка путей
db_name = 'Data Archive.db'
local_db_path = '/content/' + db_name
drive_db_path = '/content/drive/MyDrive/Data Base/' + db_name  # Путь к базе данных в Google Drive
# excel_path = '/content/drive/My Drive/Data Base/Data Archive.xlsm'  # Путь к Excel файлу

In [ ]:
# Копируем базу из Drive если она существует
if os.path.exists(drive_db_path):
  !cp "{drive_db_path}" "{local_db_path}"
  print("База данных загружена из Google Drive")
else:
  print(f"База данных {db_name} отсутствует в Google Drive")

In [ ]:
con = sqlite3.connect('Data Archive.db')
cursor = con.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()

print("\nTables:")
print("-------")
for table in tables:
    print(table[0])


for table in tables:
    table_name = table[0]
    print(f"\nТаблица: {table_name}")
    df = pd.read_sql(f"SELECT * FROM {table_name}", con)
    print(df)



# fx_curve = pd.read_sql("SELECT DISTINCT source FROM 'fx_rate'", con)
# print(fx_curve)


con.close()

In [ ]:
with closing(sqlite3.connect('Data Archive.db')) as connection:
    g = connection.cursor().execute(f"""select zero_rate from curve_quote where currency = 'N/A' and source = 'Cbonds' and date = '2025-01-09 00:00:00' and days = 'N/A' and curve = 'SOFR' """)
print(g)

In [ ]:
with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'Cbonds'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_d_l = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}' """)).fetchall()[0][0])

# **Архитектура DataArchive по таблицам (доступные источники, курсы)**

**fx_rate**

источники и доступные столбцы в них:

1.   Cbonds - px_last
2.   CBR - px_clir
3.   Foreign - px_low, px_high, px_close
4.   NCC - px_last
5.   OFFSHORE - в актуальной версии бд данных по источнику нет ни в одном из столбцов
6.   ONSHORE - px_clir
7.   RuData FOREX -  в актуальной версии бд данных по источнику нет ни в одном из столбцов
8.   RuData MOEX - px_low, px_high, px_last

**curve_quote**

источники:

1.   Cbonds - есть пропуски в данных
2.   Foreign - все данны есть
3.   RuData - тоже все ок

Для всех источников доступный столбец со значениями - zero_rate


Сейчас ставки подгружаются по фильтрам источника, валюты, тенора/дней, ставка не прописывается явно

**swap_points_fwd**

источники:

1.   Cbonds - last (единственное значение, не подходит для диапазона min/max)
2.   MOEX - low, high, last
3.   MOEX Backtrader - low, high

In [ ]:
# Если данные не парсятся и выдают ошибку по типу index out of list, то дописать в pd.to_datetime (value, format = '%Y-%m-%d')

# **Часть к расчетам не относится, сделана, чтобы проверить по отдельности качество вытягиваемых данных из базы и общие моменты**

In [ ]:
table_name = 'fx_rate'
source = 'NCC'
date1 = '2024-07-31 00:00:00'
pair = 'CNYRUB'
with closing(sqlite3.connect('Data Archive.db')) as connection:
  ncc = ((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair}' and date = '{date1}' """)).fetchall()[0][0])
ncc

In [ ]:
# ЦК НКЦ

pair_NCC_name = 'USDRUB'
date = '2025-01-03 00:00:00'
table_name = 'fx_rate'
source = 'NCC'
with closing(sqlite3.connect('Data Archive.db')) as connection:
    ncc_rate =((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair_NCC_name}' and date = '{date}' """)).fetchall()[0][0])
ncc_rate

RATE parsing (fx_rate)

похожая история на своп поинты (где-то есть px_low, px_high, где-то px_clir и они "взаимоисключающие" )

In [ ]:
reestr = pd.read_excel('reestr_new_1.xlsx')
reestr

In [ ]:
table_name = 'fx_rate'

source = 'Foreign'

date1 = '2024-08-01 00:00:00'

pair = 'CNYRUB'

with closing(sqlite3.connect('Data Archive.db')) as connection:
  fx_rate_low = ((connection.cursor().execute(f"""select px_low from '{table_name}' where source = '{source}' and pair = '{pair}' and date = '{date1}' """)).fetchall()[0][0])
  fx_rate_high = ((connection.cursor().execute(f"""select px_high from '{table_name}' where source = '{source}' and pair = '{pair}' and date = '{date1}' """)).fetchall()[0][0])

fx_rate_low

SWAP POINTS parsing (swap_points_fwd)

In [ ]:
table_name = 'swap_points_fwd'

source = 'MOEX'

date1 = '2024-07-01 00:00:00'

name = 'CNY_TODTOM'

with closing(sqlite3.connect('Data Archive.db')) as connection:
  swap_points_fwd_low = ((connection.cursor().execute(f"""select low from '{table_name}' where source = '{source}' and  name = '{name}' and date = '{date1}' """)).fetchall()[0][0])
  swap_points_fwd_high = ((connection.cursor().execute(f"""select high from '{table_name}' where source = '{source}' and  name = '{name}' and date = '{date1}' """)).fetchall()[0][0])
swap_points_fwd_low
#swap_points_fwd_high

CURVES parsing (curve_quote)

В спецификации кривых записано как 1Y,  где-то 12M (В частности AUD, USD, CHF 1Y, а TRY, CAD 12M). Например, не получится вытянуть JPY_OIS на 12M (только 1Y). Если связывать исключительно по валюте, без привязки к определенной кривой, то ок


In [ ]:
reestr_initial = pd.read_excel('reestr_new_1.xlsx')
f = reestr_initial['Покупаемая валюта1 '][0]
f

In [ ]:
curves = {
    'CNY': 'CNY_CNY',
    'USD': 'SOFR',
    'EUR': 'EUR_ESTR',
    'AUD': 'AUD AONIA ICAA',
    'CAD': 'CAD CORRA REF',
    'CHF': 'CHF SARON ICAP',
    'GBP': 'GBP SONIA TWMK',
    'HKD': 'HKD HONIA TRHK',
    'JPY': 'JPY TONAR TKFX',
    'RUB': 'RUB_RUB',
    'TRY': 'TRY TLREF REF'
    }


In [ ]:
curves['TRY']

In [ ]:
reestr_initial = pd.read_excel('reestr_new_1.xlsx')
reestr_initial['Покупаемая валюта1 '][0]

In [ ]:
def curve_id(currency_1):
        x = curves.get.reestr_initial['Покупаемая валюта1 ']
        return x

In [ ]:
curves.get(reestr_initial['Покупаемая валюта1 '][5])

In [ ]:
tenor = '3Y'

table_name = 'curve_quote'

currency = 'CNY' # currency_pair(:3)/(3:)

source = 'MOEX'

date = '2024-07-30 00:00:00'

curve = 'CNY_CNY'

from contextlib import closing

with closing(sqlite3.connect('Data Archive.db')) as connection:

  rate_domestic = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where tenor = '{tenor}' and currency = '{currency}' and source = '{source}' and date = '{date}' and curve = '{curve}' """)).fetchall()[0][0])
rate_domestic

# Reestr parsing

In [ ]:
reestr_initial = pd.read_excel('reestr_new_1.xlsx')
reestr_initial


# FUNCTIONS


In [ ]:
# Предлагаю разбить все функции на блоки

In [ ]:
# Словарь приоритезации валют
currency_priority_dict = {
    'XDR': 0,
    'EUR': 1,
    'GBP': 2,
    'AUD': 3,
    'NZD': 4,
    'USD': 5,
    'CAD': 6,
    'CHF': 7,
    'TRY': 8,
    'DKK': 9,
    'NOK': 10,
    'ZAR': 11,
    'SEK': 12,
    'CNY': 13,
    'CNH': 14,
    'BRL': 15,
    'HKD': 16,
    'INR': 17,
    'CZK': 18,
    'KZT': 19,
    'JPY': 20,
    'BYN': 21,
    'SGD': 22,
    'KRW': 23,
    'HUF': 24,
    'RON': 25,
    'AZN': 26,
    'BGN': 27,
    'UZS': 28,
    'KGS': 29,
    'MDL': 30,
    'PLN': 31,
    'TMT': 32,
    'TJS': 33,
    'UAH': 34,
    'AMD': 35,
    'GEL': 36,
    'RUB': 37
}


# находим разницу по своп пунктам через ставки
# def swap_points_simple(spot, internal_rate, external_rate, time):
#   simple = (spot*(1+(internal_rate)*time/365)/(1+(external_rate)*time/365))-spot
#   return simple



In [ ]:
# Функция определения валютной пары
def currency_pair(curr1, curr2):

    try:
        if currency_priority_dict[curr1] < currency_priority_dict[curr2]:
            pair = curr1 + curr2
        else:
            pair = curr2 + curr1
        return pair
    except KeyError:
        print("Валюта не найдена в currency_priority_dict!")

In [ ]:
# Функция определения направления сделки
def deals_dir(curr1, curr2, curr_pair):
    """
    Input: currency 1, currency 2 and currency pair as str
    Output: direction of deal according to input parametres (SELL OR BUY)
    """
    if (curr1 in curr_pair) and (curr2 in curr_pair):
        if curr_pair[:3] == curr1:
            ddir = 'SELL'
        elif curr_pair[:3] == curr2:
            ddir = 'BUY'
    else:
        print('Несоответствие валют. Проверьте входные данные!')
        ddir = None
    return ddir

In [ ]:
# Функция определения суммы базовой валюты
def sum_curr_base(curr1, curr2, curr_pair, sum_sell, sum_buy):
    """
    Input: currency 1, currency 2 and currency pair as str; sum sell and sum buy
    Output: sum of base currency
    """
    if (curr1 in curr_pair) and (curr2 in curr_pair):
        if curr_pair[:3] == curr1:
            sum_base = sum_sell
        elif curr_pair[:3] == curr2:
            sum_base = sum_buy
    else:
        print('Несоответствие валют. Проверьте входные данные!')
        sum_base = None
    return sum_base

In [ ]:
# Функция определение суммы расчетной валюты
def sum_curr_calc(curr1, curr2, curr_pair, sum_sell, sum_buy):
    """
    Input: currency 1, currency 2 and currency pair as str; sum sell and sum buy
    Output: sum of calculation currency
    """
    if (curr1 in curr_pair) and (curr2 in curr_pair):
        if curr_pair[:3] == curr2:
            sum_curr = sum_sell
        elif curr_pair[:3] == curr1:
            sum_curr = sum_buy
    else:
        print('Несоответствие валют. Проверьте входные данные!')
        sum_curr = None
    return sum_curr

In [ ]:
# Функция определения расхождения спот курсов
def diff_spot(spot_rate, spot_min_rate, spot_max_rate):
    """
    Input: spot_rate, spot_min_rate and spot_max_rate as float
    Output: spot max exchange rate on the spot_date from the specified source
    """
    if (spot_min_rate == None) or (spot_min_rate == 0) \
        or (spot_max_rate == None) or (spot_max_rate == 0):
        diff_spot = 'Нет данных'
    else:
        if (spot_rate >= spot_min_rate) and (spot_rate <= spot_max_rate):
            diff_spot = 'В рынке'
        elif spot_rate < spot_min_rate:
            diff_spot = (spot_rate - spot_min_rate) / spot_min_rate
        else:
            diff_spot = (spot_rate - spot_max_rate) / spot_max_rate
    return diff_spot

In [ ]:
# Функция определения расхождения своп-разницы
def diff_swap_points(points_bank, points_min, points_max):
    """
    Input: swap points from bank and calculated min and max as float
    Output: fwd rate as float in a simple compounding type
    """
    if points_min in (0, None, 'Нет данных') or points_max in (0, None, 'Нет данных'):
        print('NO data')
        #diff_swap = points_bank # поч при отсутствии данных по поинтам они принимаются равными банковсуким данным ???______________________________________________________
    elif (abs((points_min + points_max)/2 - points_bank) <= \
          (max(points_min, points_max) - min(points_min, points_max))/2):
        diff_swap = 'В рынке'
    elif points_bank < min(points_min, points_max):
        diff_swap = points_bank - min(points_min, points_max)
    else:
        diff_swap = points_bank - max(points_min, points_max)
    return diff_swap

In [ ]:
# Функция определения источника своп поинтов (напрямую с рынка для сделок < 3 дней или через ставки)
def type_swp(duration_of_contract):
  if duration_of_contract <= 3:
    type_swp = 'direct swp'
  else:
    type_swp = 'rates'
  return type_swp

In [ ]:
# определяем по сроку сделки больший тенор для ставок
def tenor1h(days_of_swap):
  if days_of_swap <= 7:
    tenor = 7
  if days_of_swap <=14 and days_of_swap >=7:
    tenor = 14
  if days_of_swap <=30 and days_of_swap >=14:
    tenor = 30
  if days_of_swap <= 60 and days_of_swap >=30:
    tenor = 60
  if days_of_swap <= 90 and days_of_swap >=60:
    tenor = 90
  if days_of_swap <= 180 and days_of_swap >=90:
    tenor = 180
  if days_of_swap <= 270 and days_of_swap >=180:
    tenor = 270
  if days_of_swap <= 360 and days_of_swap >=270:
    tenor = 360
  return tenor

# определяем по сроку сделки меньший тенор для ставок
def tenor2l(days_of_swap):
  if days_of_swap <= 7:
    tenor = 7
  if days_of_swap <=14 and days_of_swap >=7:
    tenor = 7
  if days_of_swap <=30 and days_of_swap >=14:
    tenor = 14
  if days_of_swap <= 60 and days_of_swap >=30:
    tenor = 30
  if days_of_swap <= 90 and days_of_swap >=60:
    tenor = 60
  if days_of_swap <= 180 and days_of_swap >=90:
    tenor = 90
  if days_of_swap <= 270 and days_of_swap >=180:
    tenor = 180
  if days_of_swap <= 360 and days_of_swap >= 270:
    tenor = 360
  return tenor

In [ ]:
curves = {
    'CNY': 'CNY_CNY',
    'USD': 'SOFR',
    'EUR': 'EUR_ESTR',
    'AUD': 'AUD AONIA ICAA',
    'CAD': 'CAD CORRA REF',
    'CHF': 'CHF SARON ICAP',
    'GBP': 'GBP SONIA TWMK',
    'HKD': 'HKD HONIA TRHK',
    'JPY': 'JPY TONAR TKFX',
    'RUB': 'RUB_RUB',
    'TRY': 'TRY TLREF REF'
    }


def curve_ident(currency_1):
        x = curves.get(currency_1)
        return x

In [ ]:
def rate_domestic_pars_l(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'MOEX'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_d_l = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}'""")).fetchall()[0][0])
  return rate_d_l

In [ ]:
rate_domestic_pars_l('CNY', '2024-07-31 00:00:00', '7', 'CNY_CNY')

In [ ]:
#находим внутренюю ставку меньшую
def rate_domestic_pars_l(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'MOEX'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_d_l = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}' """)).fetchall()[0][0])
  return rate_d_l

#находим внутренюю ставку большую
def rate_domestic_pars_h(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'MOEX'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_d_h = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}' """)).fetchall()[0][0])
  return rate_d_h

#находим внешнюю ставку меньшую
def rate_foreign_pars_l(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'MOEX'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_f_l = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}'and curve = '{curve}' """)).fetchall()[0][0])
  return rate_f_l

#находим внешнюю ставку большую
def rate_foreign_pars_h(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'MOEX'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_f_h = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}' """)).fetchall()[0][0])
  return rate_f_h

In [ ]:
# def sum_currency(curr1, curr2, curr_pair, sum_sell, sum_buy, currency_type='base'):

#     if (curr1 not in curr_pair) or (curr2 not in curr_pair):
#         print('Ошибка: Несоответствие валют. Проверьте входные данные!')
#         return None

#     base_curr = curr_pair[:3]

#     if currency_type == 'base':
#         return sum_sell if base_curr == curr1 else sum_buy
#     elif currency_type == 'calc':
#         return sum_buy if base_curr == curr1 else sum_sell
#     else:
#         print('Ошибка: Неверный тип валюты. Используйте "base" или "calc".')
#         return None

In [ ]:
def rate_domestic_pars_l(currency, date, days, curve):
  from contextlib import closing
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    table_name = 'curve_quote'
    source = 'Cbonds'
    #source = input("print desired source of info (Cbonds, Foreign, MARKIT, MOEX, RuData): ")
    rate_d_l = ((connection.cursor().execute(f"""select zero_rate from '{table_name}' where currency = '{currency}' and source = '{source}' and date = '{date}' and days = '{days}' and curve = '{curve}' """)).fetchall()[0][0])
  return rate_d_l


rate_domestic_pars_l('N/A', '2025-01-09', 'N/A', 'SOFR')

In [ ]:
g = connection.cursor().execute(f"""select zero_rate from curve_quote where currency = 'N/A' and source = 'Cbonds' and date = '2025-01-09 00:00:00' and days = 'N/A' and curve = 'SOFR' """)

# Блок для проверки рыночных данных на наличие (все функции уже к этому моменту определены, прежде чем вытаскивать в окончательный реестр, сделаем блок с проверкой)

In [ ]:
def exist(date, table, currency_pair, source):
    """
    Input: date, name of table from database, currency pair for parsing, source (according to table)
    """
    date = pd.to_datetime(date, dayfirst = True)

    try:
        with closing(sqlite3.connect('Data Archive.db')) as connection:
            ncc = ((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair}' and date = '{date}' """)).fetchall()[0][0])
    except KeyError:
        print("Данных по валюте нет")
    return ncc

In [ ]:
exist(reestr_initial['Дата платежа1'][0], 'fx_rate', 'CNYRUB', 'NCC')

In [ ]:
reestr_mini = pd.read_excel('reestr_new_1.xlsx')
reestr_mini

# CALCULATE REESTR

In [ ]:
reestr_initial = pd.read_excel('reestr_new_1.xlsx')

**The most actual version**

In [ ]:
reestr_mini = pd.read_excel('reestr_new_1.xlsx')
reestr_mini['Дата оценки'] = reestr_mini['Дата платежа1']
reestr_mini ['Продаваемая Банком валюта'] = reestr_mini['Продаваемая валюта1']
reestr_mini ['Покупаемая Банком валюта'] = reestr_mini['Покупаемая валюта1 ']
reestr_mini['Валюта расчетов'] = input("валюта расчетов: ")

# временный костыль с которым все работает, без него на дату оценки тянутся какие-то левые курсы
###reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'] = '2024-08-01'
#

# Вспомогательная часть длю валютных пар
reestr_mini['Валютная пара 1'] = reestr_mini['Покупаемая валюта2 '] + reestr_mini['Валюта расчетов']
reestr_mini['Валютная пара 2'] = reestr_mini['Покупаемая валюта1 '] + reestr_mini['Валюта расчетов']


# Валютная пара
curr_pair = []
for i in range(len(reestr_mini['№ сделки'])):
  curr_pair.append(currency_pair(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i]))
reestr_mini['Валютная пара'] = curr_pair


# Направление сделки
direction = []
for i in range(len(reestr_mini['№ сделки'])):
  direction.append(deals_dir(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i]))
reestr_mini ['Направление сделки'] = direction
reestr_mini ['Название'] = reestr_mini['Валютная пара'] + reestr_mini['Направление сделки']


# Сумма в базовой валюте
sum_base = []
for i in range(len(reestr_mini['№ сделки'])):
  sum_base.append(sum_curr_base(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i], reestr_mini['Сумма продажи1'][i], reestr_mini['Сумма покупки1'][i]))
reestr_mini['Сумма в базовой валюте'] =  sum_base


# Сумма в расчетной валюте
sum_calc = []
for i in range(len(reestr_mini['№ сделки'])):
  sum_calc.append(sum_curr_calc(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i], reestr_mini['Сумма продажи1'][i], reestr_mini['Сумма покупки1'][i]))
reestr_mini['Сумма в расчетной валюте'] = sum_calc

reestr_mini['Курс спот сделки (1 части)'] = reestr_mini['Курс1']


# ЦК НКЦ
ncc_rate = []
for i in range(len(reestr_mini['№ сделки'])):
  pair_NCC_name = reestr_mini['Валютная пара'][i]
  date = (pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True))
  table_name = 'fx_rate'
  source = 'NCC'
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      ncc_rate.append((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair_NCC_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['ЦК НКЦ'] = ncc_rate


# Спот мин на дату сделки
spot_min = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_min.append((connection.cursor().execute(f"""select px_low from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот мин'] = spot_min


# spot min of valuation date
spot_min_val_date = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'][i], dayfirst = True) #!!!
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_min_val_date.append((connection.cursor().execute(f"""select px_low from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот мин val date'] = spot_min_val_date


# Спот макс на дату сделки
spot_max = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_max.append((connection.cursor().execute(f"""select px_high from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот макс'] = spot_max


# Спот макс val date
spot_max_val_date = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'][i], dayfirst = True) #!!!
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_max_val_date.append((connection.cursor().execute(f"""select px_high from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот макс val date'] = spot_max_val_date


# Разница спот
spot_diff = []
for i in range(len(reestr_mini['№ сделки'])):
  spot_diff.append(diff_spot(reestr_mini['Курс спот сделки (1 части)'][i], reestr_mini['Спот мин'][i], reestr_mini['Спот макс'][i]))
reestr_mini['Расхождение'] =  spot_diff


# Срок, дней
days = []
for i in range(len(reestr_mini['№ сделки'])):
  a = pd.to_datetime(reestr_mini['Дата платежа2'], dayfirst=True) - pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'], dayfirst=True)
  days = a.dt.days
reestr_mini['Срок'] = days


# Тип своп-поинтов (ставки/рынок)
type = []
for i in range(len(reestr_mini['№ сделки'])):
  type.append(type_swp(reestr_mini['Срок'][i]))
reestr_mini ['тип'] = type


# Своп-разница по сделке
swap_diff = []
for i in range(len(reestr_mini['№ сделки'])):
  swap_diff.append(reestr_mini['Курс2'][i] - reestr_mini['Курс1'][i])
reestr_mini['Своп-разница по сделке'] = swap_diff


# Своп-разница мин direct swappoints
swp_min = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'swap_points_fwd'
  source = 'MOEX'
  pair_swap_name = reestr_mini['Валютная пара'][i][:3] + "_TODTOM"
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    swp_min.append((connection.cursor().execute(f""" select low from '{table_name}' where source = '{source}' and name = '{pair_swap_name}' and date = '{date}' """)).fetchall()[0][0])
#reestr_mini['Своп-разница мин'] = swp_min


# Своп-разница макс
swp_max = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'swap_points_fwd'
  source = 'MOEX'
  pair_swap_name = reestr_mini['Валютная пара'][i][:3] + "_TODTOM"
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    swp_max.append((connection.cursor().execute(f""" select high from '{table_name}' where source = '{source}' and name = '{pair_swap_name}' and date = '{date}' """)).fetchall()[0][0])
#reestr_mini['Своп-разница макс'] = swp_max

cur_1 = []
for i in range(len(reestr_mini['№ сделки'])):
  cur_1 = reestr_mini['Валютная пара'][i][3:]
reestr_mini['Валюта 1'] = cur_1


cur_2 = []
for i in range(len(reestr_mini['№ сделки'])):
  cur_2 = reestr_mini['Валютная пара'][i][:3]
reestr_mini['Валюта 2'] = cur_2


tenor_low =[]
for i in range(len(reestr_mini['№ сделки'])):
    tenor_low.append(tenor2l(reestr_mini['Срок'][i]))
reestr_mini['Тенор мин'] = tenor_low


tenor_high =[]
for i in range(len(reestr_mini['№ сделки'])):
    tenor_high.append(tenor1h(reestr_mini['Срок'][i]))
reestr_mini['Тенор макс'] = tenor_high


##### в идеале сделать так, чтобы словарь с валютами и ставками автоматом подставлялся в запрос в тип ставки UPD: уже реализовано
curve_id_1 = []
for i in range(len(reestr_mini['№ сделки'])):
    curve_id_1.append(curve_ident(reestr_mini['Продаваемая Банком валюта'][i]))
reestr_mini ['ставка_1'] = curve_id_1

curve_id_2 = []
for i in range(len(reestr_mini['№ сделки'])):
    curve_id_2.append(curve_ident(reestr_mini['Покупаемая Банком валюта'][i]))
reestr_mini ['ставка_2'] = curve_id_2


# Вспомогательные строки для ставок

rates_domestic_l = []
for i in range(len(reestr_mini['№ сделки'])):
  rates_domestic_l.append(rate_domestic_pars_l(reestr_mini['Валюта 1'][i], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[i], reestr_mini['Тенор мин'][i], reestr_mini['ставка_1'][i]))
reestr_mini['Ставка внутреняя мин']= rates_domestic_l

rates_domestic_h = []
for i in range(len(reestr_mini['№ сделки'])):
  rates_domestic_h.append(rate_domestic_pars_h(reestr_mini['Валюта 1'][i], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[i], reestr_mini['Тенор макс'][i], reestr_mini['ставка_1'][i]))
reestr_mini['Ставка внутреняя макс']= rates_domestic_h

rates_foreign_l = []
for i in range(len(reestr_mini['№ сделки'])):
  rates_foreign_l.append(rate_foreign_pars_l(reestr_mini['Валюта 2'][i], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[i], reestr_mini['Тенор мин'][i], reestr_mini['ставка_2'][i]))
reestr_mini['Ставка внешняя мин']= rates_foreign_l

rates_foreign_h = []
for i in range(len(reestr_mini['№ сделки'])):
  rates_foreign_h.append(rate_foreign_pars_h(reestr_mini['Валюта 2'][i], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[i], reestr_mini['Тенор макс'][i], reestr_mini['ставка_2'][i]))
reestr_mini['Ставка внешняя макс']= rates_foreign_h
reestr_mini['Интер ставка внутреняя'] = reestr_mini.apply(lambda row: np.interp(row['Срок'], [row['Тенор мин'], row['Тенор макс']], [row['Ставка внутреняя мин'], row['Ставка внутреняя макс']]), axis=1)

reestr_mini['Интер ставка внешняя'] = reestr_mini.apply(lambda row: np.interp(row['Срок'], [row['Тенор мин'], row['Тенор макс']], [row['Ставка внешняя мин'], row['Ставка внешняя макс']]), axis=1)

# своп поинты через ставки low
# previous reestr_mini['Своп-поинты мин_ставка'] = reestr_mini.apply(lambda row: ((row['Спот мин']*(1+row['Интер ставка внутреняя']*row['Срок']/365)/(1+row['Интер ставка внешняя']*row['Срок']/365))-row['Спот мин']), axis =1)
reestr_mini['Своп-поинты мин_ставка'] = reestr_mini.apply(lambda row: ((row['Спот мин']*((1+row['Интер ставка внутреняя']/100)/(1+row['Интер ставка внешняя']/100))**(row['Срок']/365)) - row['Спот мин']), axis =1)

# своп поинты через ставки high
#reestr_mini['Своп-поинты макс_ставка'] = reestr_mini.apply(lambda row: ((row['Спот макс']*(1+row['Интер ставка внутреняя']*row['Срок']/365)/(1+row['Интер ставка внешняя']*row['Срок']/365))-row['Спот макс']), axis =1)
reestr_mini['Своп-поинты макс_ставка'] = reestr_mini.apply(lambda row: ((row['Спот макс']*((1+row['Интер ставка внутреняя']/100)/(1+row['Интер ставка внешняя']/100))**(row['Срок']/365)) - row['Спот макс']), axis =1)


# развилка по своп поинтам

for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['тип'][i] == 'direct swp':
    reestr_initial['Своп-разница min real'] = swp_min
    reestr_initial['Своп-разница max real'] = swp_max
  else:
    reestr_initial['Своп-разница min real'] = reestr_mini['Своп-поинты мин_ставка']
    reestr_initial['Своп-разница max real'] = reestr_mini['Своп-поинты макс_ставка']

reestr_mini['swap max'] = reestr_initial['Своп-разница max real']
reestr_mini['swap min'] = reestr_initial['Своп-разница min real']


# Расхождение своп через прямые своп-пункты. Если тип direct, то эта функция в итоговый реестр, иначе пойдет через ставки именно сама своп-разница
diff_swap = []
for i in range(len(reestr_mini['№ сделки'])):
  diff_swap.append(diff_swap_points(reestr_mini['Своп-разница по сделке'][i], reestr_mini['swap min'][i], reestr_mini['swap max'][i]))
reestr_mini['Расхождение своп'] = diff_swap



# При отсутствии расхождение получается, что в ячейке "расхождение своп" = в рынке, и при делении возникает ошибка. Может сделаем этот столбец при условии, что расхождение содержит численные значения (то есть не в рынке)
# #% расхождение в %

for i in range(len(reestr_mini['№ сделки'])):
    if reestr_mini['Расхождение своп'][i] == 'В рынке':
        reestr_mini ['Расхождение своп %'] = '-'
    else:
        reestr_mini ['Расхождение своп %'] = (reestr_mini['Расхождение своп'][i] / reestr_mini['Спот макс'][i])


# swap_points_percent = []
# for i in range(len(reestr_mini['Продаваемая Банком валюта'])):
#     try:
#         swap_points_percent.append(reestr_mini['Расхождение своп'][i] / reestr_mini['Спот макс'][i])
#     except TypeError:
#         swap_points_percent.append('-')
# reestr_mini ['Расхождение своп %'] = swap_points_percent

# #
# swap_points_percent = []
# for i in range(len(reestr_mini['Продаваемая Банком валюта'])):

#     swap_points_percent.append(reestr_mini['Расхождение своп'][i] / reestr_mini['Спот макс'][i])

# reestr_mini ['Расхождение своп %'] = swap_points_percent
#

#надо добавить источник ставок

# discount factor in vs ex
DF_1 = []
for i in range(len(reestr_mini['№ сделки'])):
  DF_1.append(1/(1+(reestr_mini['Интер ставка внутреняя'][i]) * (reestr_mini['Срок'][i])/365))
reestr_mini['DF_1']= DF_1

DF_2 = []
for i in range(len(reestr_mini['№ сделки'])):
  DF_2.append(1/(1+(reestr_mini['Интер ставка внешняя'][i]) * (reestr_mini['Срок'][i])/365))
reestr_mini['DF 2']= DF_2


# вспомогательная часть для курсов для СС
val_1l = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 1'][i]  == 'RUBRUB':
    val_1l.append(1)
  else:
    val_1l.append(reestr_mini['Спот мин val date'][i])
reestr_mini['Валютная пара 1, мин на дату оценки'] = val_1l

val_1h = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 1'][i]  == 'RUBRUB':
    val_1h.append(1)
  else:
    val_1h.append(reestr_mini['Спот макс val date'][i])
reestr_mini['Валютная пара 1, макс на дату оценки'] = val_1h


val_2l = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 2'][i]  == 'RUBRUB':
    val_2l.append(1)
  else:
    val_2l.append(reestr_mini['Спот мин val date'][i])
reestr_mini['Валютная пара 2, мин на дату оценки'] = val_2l

val_2h = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 2'][i]  == 'RUBRUB':
    val_2h.append(1)
  else:
    val_2h.append(reestr_mini['Спот макс val date'][i])
reestr_mini['Валютная пара 2, макс на дату оценки'] = val_2h


# курс цб
CB_RF = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'CBR'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      CB_RF.append((connection.cursor().execute(f"""select px_clir from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['курс ЦБ'] = CB_RF


# курсы для фв
cbr_1 = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 1'][i]  == 'RUBRUB':
    cbr_1.append(1)
  else:
    cbr_1.append(reestr_mini['курс ЦБ'][i])
reestr_mini['cbr_1'] = cbr_1


cbr_2 = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_mini['Валютная пара 2'][i]  == 'RUBRUB':
    cbr_2.append(1)
  else:
    cbr_2.append(reestr_mini['курс ЦБ'][i])
reestr_mini['cbr_2'] = cbr_2


# FV part
FVL = []
for i in range(len(reestr_mini['№ сделки'])):
  FVL.append((reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['Валютная пара 1, мин на дату оценки'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['Валютная пара 2, макс на дату оценки'][i])/1000)
reestr_mini['Справедливая стоимость на дату оценки low']= FVL


FVH = []
for i in range(len(reestr_mini['№ сделки'])):
  FVH.append((reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['Валютная пара 1, макс на дату оценки'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['Валютная пара 2, мин на дату оценки'][i])/1000)
reestr_mini['Справедливая стоимость на дату оценки high']= FVH


FVCB = []
for i in range(len(reestr_mini['№ сделки'])):
  FVCB.append((reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['cbr_1'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['cbr_2'][i])/1000)
reestr_mini['Справедливая стоимость на дату оценки по курсу ЦБ'] = FVCB


# расхождение с СС Банка (часть из Python_Swap_Beta)

fv_delta_abs = []
fv_delta_percent = []
for i in range(len(reestr_mini['№ сделки'])):
  if reestr_initial['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] < reestr_mini['Справедливая стоимость на дату оценки low'][i]:
    fv_delta_abs.append(reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. '][i] - reestr_mini['Справедливая стоимость на дату оценки low'][i])
    fv_delta_percent.append((reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки low'][i])*1000 / reestr_mini['Сумма в расчетной валюте'][i])
  elif reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] > reestr_mini['Справедливая стоимость на дату оценки high'][i]:
    fv_delta_abs.append(reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки high'][i])
    fv_delta_percent.append((reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки high'][i])*1000 / reestr_mini['Сумма в расчетной валюте'][i])
  else:
    fv_delta_abs.append("В рынке")
    fv_delta_percent.append(0.)

reestr_mini.head()

In [ ]:
reestr_mini = pd.read_excel('reestr_new_1.xlsx')
reestr_mini['Дата оценки'] = reestr_mini['Дата платежа1']
reestr_mini ['Продаваемая Банком валюта'] = reestr_mini['Продаваемая валюта1']
reestr_mini ['Покупаемая Банком валюта'] = reestr_mini['Покупаемая валюта1 ']
reestr_mini['Валюта расчетов'] = input("валюта расчетов: ")

# временный костыль с которым все работает, без него на дату оценки тянутся какие-то левые курсы
###reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'] = '2024-08-01'
#

# Вспомогательная часть длю валютных пар
reestr_mini['Валютная пара 1'] = reestr_mini['Покупаемая валюта2 '] + reestr_mini['Валюта расчетов']
reestr_mini['Валютная пара 2'] = reestr_mini['Покупаемая валюта1 '] + reestr_mini['Валюта расчетов']


# Валютная пара
curr_pair = []
for i in range(len(reestr_mini['№ сделки'])):
  curr_pair.append(currency_pair(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i]))
reestr_mini['Валютная пара'] = curr_pair


# Направление сделки
direction = []
for i in range(len(reestr_mini['№ сделки'])):
  direction.append(deals_dir(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i]))
reestr_mini ['Направление сделки'] = direction
reestr_mini ['Название'] = reestr_mini['Валютная пара'] + reestr_mini['Направление сделки']


# Сумма в базовой валюте
sum_base = []
for i in range(len(reestr_mini['№ сделки'])):
  sum_base.append(sum_curr_base(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i], reestr_mini['Сумма продажи1'][i], reestr_mini['Сумма покупки1'][i]))
reestr_mini['Сумма в базовой валюте'] =  sum_base


# Сумма в расчетной валюте
sum_calc = []
for i in range(len(reestr_mini['№ сделки'])):
  sum_calc.append(sum_curr_calc(reestr_mini['Продаваемая Банком валюта'][i], reestr_mini['Покупаемая Банком валюта'][i], reestr_mini['Валютная пара'][i], reestr_mini['Сумма продажи1'][i], reestr_mini['Сумма покупки1'][i]))
reestr_mini['Сумма в расчетной валюте'] = sum_calc

reestr_mini['Курс спот сделки (1 части)'] = reestr_mini['Курс1']


# ЦК НКЦ
ncc_rate = []
for i in range(len(reestr_mini['№ сделки'])):
  pair_NCC_name = reestr_mini['Валютная пара'][i]
  date = (pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True))
  table_name = 'fx_rate'
  source = 'NCC'
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      ncc_rate.append((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair_NCC_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['ЦК НКЦ'] = ncc_rate


# Спот мин на дату сделки
spot_min = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_min.append((connection.cursor().execute(f"""select px_low from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот мин'] = spot_min


# spot min of valuation date
spot_min_val_date = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'][i], dayfirst = True) #!!!
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_min_val_date.append((connection.cursor().execute(f"""select px_low from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот мин val date'] = spot_min_val_date


# Спот макс на дату сделки
spot_max = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_max.append((connection.cursor().execute(f"""select px_high from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот макс'] = spot_max


# Спот макс val date
spot_max_val_date = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'fx_rate'
  source = 'Foreign'
  pair_spot_name = reestr_mini['Валютная пара'][i]
  date = pd.to_datetime(reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ'][i], dayfirst = True) #!!!
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      spot_max_val_date.append((connection.cursor().execute(f"""select px_high from '{table_name}' where source = '{source}' and pair = '{pair_spot_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['Спот макс val date'] = spot_max_val_date


# Разница спот
spot_diff = []
for i in range(len(reestr_mini['№ сделки'])):
  spot_diff.append(diff_spot(reestr_mini['Курс спот сделки (1 части)'][i], reestr_mini['Спот мин'][i], reestr_mini['Спот макс'][i]))
reestr_mini['Расхождение'] =  spot_diff


# Срок, дней
days = []
for i in range(len(reestr_mini['№ сделки'])):
  a = pd.to_datetime(reestr_mini['Дата платежа2'], dayfirst=True) - pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'], dayfirst=True)
  days = a.dt.days
reestr_mini['Срок'] = days


# Тип своп-поинтов (ставки/рынок)
type = []
for i in range(len(reestr_mini['№ сделки'])):
  type.append(type_swp(reestr_mini['Срок'][i]))
reestr_mini ['тип'] = type


# Своп-разница по сделке
swap_diff = []
for i in range(len(reestr_mini['№ сделки'])):
  swap_diff.append(reestr_mini['Курс2'][i] - reestr_mini['Курс1'][i])
reestr_mini['Своп-разница по сделке'] = swap_diff


# Своп-разница мин direct swappoints
swp_min = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'swap_points_fwd'
  source = 'MOEX'
  pair_swap_name = reestr_mini['Валютная пара'][i][:3] + "_TODTOM"
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    swp_min.append((connection.cursor().execute(f""" select low from '{table_name}' where source = '{source}' and name = '{pair_swap_name}' and date = '{date}' """)).fetchall()[0][0])
#reestr_mini['Своп-разница мин'] = swp_min


# Своп-разница макс
swp_max = []
for i in range(len(reestr_mini['№ сделки'])):
  table_name = 'swap_points_fwd'
  source = 'MOEX'
  pair_swap_name = reestr_mini['Валютная пара'][i][:3] + "_TODTOM"
  date = pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True)
  with closing(sqlite3.connect('Data Archive.db')) as connection:
    swp_max.append((connection.cursor().execute(f""" select high from '{table_name}' where source = '{source}' and name = '{pair_swap_name}' and date = '{date}' """)).fetchall()[0][0])
#reestr_mini['Своп-разница макс'] = swp_max

cur_1 = []
for i in range(len(reestr_mini['№ сделки'])):
  cur_1 = reestr_mini['Валютная пара'][i][3:]
reestr_mini['Валюта 1'] = cur_1


cur_2 = []
for i in range(len(reestr_mini['№ сделки'])):
  cur_2 = reestr_mini['Валютная пара'][i][:3]
reestr_mini['Валюта 2'] = cur_2


tenor_low =[]
for i in range(len(reestr_mini['№ сделки'])):
    tenor_low.append(tenor2l(reestr_mini['Срок'][i]))
reestr_mini['Тенор мин'] = tenor_low


tenor_high =[]
for i in range(len(reestr_mini['№ сделки'])):
    tenor_high.append(tenor1h(reestr_mini['Срок'][i]))
reestr_mini['Тенор макс'] = tenor_high


##### в идеале сделать так, чтобы словарь с валютами и ставками автоматом подставлялся в запрос в тип ставки UPD: уже реализовано
curve_id_1 = []
for i in range(len(reestr_mini['№ сделки'])):
    curve_id_1.append(curve_ident(reestr_mini['Продаваемая Банком валюта'][i]))
reestr_mini ['ставка_1'] = curve_id_1

curve_id_2 = []
for i in range(len(reestr_mini['№ сделки'])):
    curve_id_2.append(curve_ident(reestr_mini['Покупаемая Банком валюта'][i]))
reestr_mini ['ставка_2'] = curve_id_2


reestr_mini

In [ ]:
# rates_domestic_l = []
# for i in range(len(reestr_mini['№ сделки'])):
#   rates_domestic_l.append(rate_domestic_pars_l(reestr_mini['Валюта 1'][i], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[i], reestr_mini['Тенор мин'][i], reestr_mini['ставка_1'][i]))
# reestr_mini['Ставка внутреняя мин']= rates_domestic_l



rates_domestic_l(rate_domestic_pars_l(reestr_mini['Валюта 1'][1], pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'])[1], reestr_mini['Тенор мин'][1], reestr_mini['ставка_1'][1]))


In [ ]:
reestr_mini.to_excel('k.xlsx')

Курсы выгружаются на первые 153 строчки((( остальные 200 для кода какая-то шутка

In [ ]:
len(spot_min)

In [ ]:
ncc_rate

In [ ]:
 pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][0], dayfirst = True)

 reestr_mini['Валютная пара'][0]

In [ ]:
pair_NCC_name = reestr_mini['Валютная пара'][0]
date =  pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][0], dayfirst = True)             #'2025-01-09 00:00:00'
table_name = 'fx_rate'
source = 'NCC'
with closing(sqlite3.connect('Data Archive.db')) as connection:
    ncc_rate =((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair_NCC_name}' and date = '{date}' """)).fetchall()[0][0])
ncc_rate

In [ ]:
ncc_rate = []
for i in range(len(reestr_mini['Контрагент'])):
  pair_NCC_name = reestr_mini['Валютная пара'][i]
  date = (pd.to_datetime(reestr_mini['Дата сделки, дата пролонгации/ изменения условий'][i], dayfirst = True))
  table_name = 'fx_rate'
  source = 'NCC'
  with closing(sqlite3.connect('Data Archive.db')) as connection:
      ncc_rate.append((connection.cursor().execute(f"""select px_last from '{table_name}' where source = '{source}' and pair = '{pair_NCC_name}' and date = '{date}' """)).fetchall()[0][0])
reestr_mini['ЦК НКЦ'] = ncc_rate

## **Some functions of calculation FV**

In [ ]:
FVL = []
for i in range(len(reestr_mini['Продаваемая Банком валюта'])):
  FVL.append(reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['Валютная пара 1, мин на дату оценки'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['Валютная пара 2, макс на дату оценки'][i])
reestr_mini['Справедливая стоимость на дату оценки low']= FVL


FVH = []
for i in range(len(reestr_mini['Продаваемая Банком валюта'])):
  FVH.append(reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['Валютная пара 1, макс на дату оценки'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['Валютная пара 2, мин на дату оценки'][i])
reestr_mini['Справедливая стоимость на дату оценки high']= FVH


FVCB = []
for i in range(len(reestr_mini['Продаваемая Банком валюта'])):
  FVCB.append(reestr_mini['Сумма покупки2'][i] * reestr_mini['DF_1'][i] * reestr_mini['cbr_1'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['cbr_2'][i])
reestr_mini['Справедливая стоимость на дату оценки по курсу ЦБ'] = (FVCB)

In [ ]:
def FV_calc(nominal_1, nominal_2, df_1, df_2, fx_rate_1, fx_rate_2):
    """
    Input: nominal 1 & 2, df 1 & 2 and fx rates 1 & 2 as float and int
    Output: Fair value, 1 -- 2 = FV
    """
    if (df_1 in (0, None, 'Нет данных')) or (df_2 in (0, None, 'Нет данных')) \
        or (fx_rate_1 in (0, None, 'Нет данных')) or (fx_rate_2 in (0, None, 'Нет данных')):
        FV_value = 'Нет данных'
    else:
        FV_value = (nominal_1 * df_1 * fx_rate_1 - nominal_2 * df_2 * fx_rate_2)/1000
    return FV_value

In [ ]:
# Нахождение справедливой стоимости min
fv_value_min = []
for i in range(len(bank_sell_curr)):
#     print(i+1, 'MIN:', bank_buy_nominal_1[i], bank_sell_nominal_1[i], fv_df_values_base[i], fv_df_values_calc[i], \
#                                fv_buy_spot_min_rate[i], fv_sell_spot_min_rate[i])
    fv_value_min.append(FV_calc(bank_buy_nominal_1[i], bank_sell_nominal_1[i], fv_df_values_base[i], fv_df_values_calc[i], \
                               fv_buy_spot_min_rate[i], fv_sell_spot_max_rate[i]))

In [ ]:
# Нахождение справедливой стоимости max
fv_value_max = []
for i in range(len(bank_sell_curr)):
    #print('MAX', bank_buy_nominal_1[i], bank_sell_nominal_1[i], fv_df_values_base[i], fv_df_values_calc[i], \
    #                          fv_buy_spot_max_rate[i], fv_sell_spot_max_rate[i])
    fv_value_max.append(FV_calc(bank_buy_nominal_1[i], bank_sell_nominal_1[i], fv_df_values_base[i], fv_df_values_calc[i], \
                               fv_buy_spot_max_rate[i], fv_sell_spot_min_rate[i]))

In [ ]:
reestr_mini['Сумма покупки2'][0]* reestr_mini['DF_1'][i] * reestr_mini['Валютная пара 1, макс на дату оценки'][i] - reestr_mini['Сумма продажи2'][i] * reestr_mini['DF 2'][i] * reestr_mini['Валютная пара 2, мин на дату оценки'][i])

In [ ]:
reestr_mini['Своп-поинты мин_ставка'] = reestr_mini.apply(lambda row: ((row['Спот мин']*(1+row['Интер ставка внутреняя']*row['Срок']/365)/(1+row['Интер ставка внешняя']*row['Срок']/365))-row['Спот мин']), axis =1)
reestr_mini['Своп-поинты мин_ставка'][0]

In [ ]:
reestr_mini['Своп-поинты макс_ставка'] = reestr_mini.apply(lambda row: ((row['Спот макс']*(1+row['Интер ставка внутреняя']*row['Срок']/365)/(1+row['Интер ставка внешняя']*row['Срок']/365))-row['Спот макс']), axis =1)
reestr_mini['Своп-поинты макс_ставка'][0]

In [ ]:
u = reestr_mini.apply(lambda row: ((row['Спот макс']*((1+row['Интер ставка внутреняя']/100)/(1+row['Интер ставка внешняя']/100))**(row['Срок']/365)) - row['Спот макс']), axis =1)
u

FV

In [ ]:
# FV

reestr_mini['FV_low'] = reestr_mini.apply(
    lambda row: FV_low(
        row['Покупаемая валюта2'],
        row['Сумма покупки2'],
        row['DF_1'],
        row['DF 2'],
        row['Сумма продажи2'],
        row['Спот мин val date']
    ),
    axis=1
)

# apply будет долго отрабатывать на больших данных, лучше применять повекторное перемножение по типу reestr_mini['FV_low'] = reestr_mini['Сумма покупки2'] * reestr_mini['Спот мин val date'], можно даже без функции

In [ ]:
# cc Банка "-"

fv_delta_abs = []
fv_delta_percent = []
for i in range(len(reestr_mini['Продаваемая Банком валюта'])):
  if reestr_initial['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] < reestr_mini['Справедливая стоимость на дату оценки low'][i]:
    fv_delta_abs.append(reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. '][i] - reestr_mini['Справедливая стоимость на дату оценки low'][i])
    fv_delta_percent.append((reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. \n(Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки low'][i])*1000 / reestr_mini['Сумма в расчетной валюте'][i])
  elif reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] > reestr_mini['Справедливая стоимость на дату оценки high'][i]:
    fv_delta_abs.append(reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки high'][i])
    fv_delta_percent.append((reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][i] - reestr_mini['Справедливая стоимость на дату оценки high'][i])*1000 / reestr_mini['Сумма в расчетной валюте'][i])
  else:
    fv_delta_abs.append("В рынке")
    fv_delta_percent.append(0.)


# FINAL REESTR

In [ ]:
# Функция расчета справедливой стоимости
def FV_calc(nominal_1, nominal_2, df_1, df_2, fx_rate_1, fx_rate_2):
    """
    Input: nominal 1 & 2, df 1 & 2 and fx rates 1 & 2 as float and int
    Output: Fair value, 1 -- 2 = FV
    """
    if (df_1 in (0, None, 'Нет данных')) or (df_2 in (0, None, 'Нет данных')) \
        or (fx_rate_1 in (0, None, 'Нет данных')) or (fx_rate_2 in (0, None, 'Нет данных')):
        FV_value = 'Нет данных'
    else:
        FV_value = (nominal_1 * df_1 * fx_rate_1 - nominal_2 * df_2 * fx_rate_2)/1000
    return FV_value

In [ ]:
reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)'][0]

In [ ]:
reestr_initial = pd.read_excel('reestr_new_1.xlsx')

In [ ]:
reestr_initial['Продаваемая банком валюта'] = reestr_mini['Продаваемая Банком валюта']
reestr_initial['Покупаемая банком валюта'] = reestr_mini['Покупаемая Банком валюта']
reestr_initial['Валютная пара'] = reestr_mini['Валютная пара']
reestr_initial['Направление сделки (первой части сделки)'] = reestr_mini['Направление сделки']
reestr_initial['Название'] = reestr_mini['Название']
reestr_initial['Сумма в базовой валюте'] = reestr_mini['Сумма в базовой валюте']
reestr_initial['Сумма в расчетной валюте'] = reestr_mini['Сумма в расчетной валюте']
reestr_initial['Срок'] = reestr_mini['Срок']
reestr_initial['Курс спот сделки'] = reestr_mini['Курс спот сделки (1 части)']
reestr_initial['ЦК НКЦ'] = reestr_mini['ЦК НКЦ']
reestr_initial['Спот курс min'] = reestr_mini['Спот мин']
reestr_initial['Спот курс max'] = reestr_mini['Спот макс']
reestr_initial['Расхождение'] = reestr_mini['Расхождение']
reestr_initial['Своп-разница по сделке'] = reestr_mini['Своп-разница по сделке']
reestr_initial['Своп-разница min'] = reestr_mini['swap min']
reestr_initial['Своп-разница max'] = reestr_mini['swap max']
reestr_initial['Расхождение'] = reestr_mini['Расхождение своп']
reestr_initial['Справедливая стоимость Банка'] = reestr_mini['Справедливая стоимость на текущую дату, в тыс. руб. (Только для ПФИ)']
reestr_initial['Справедливая стоимость Службы, LOW'] = reestr_mini['Справедливая стоимость на дату оценки low']
reestr_initial['Справедливая стоимость Службы, HIGH'] = reestr_mini['Справедливая стоимость на дату оценки high']
reestr_initial['Справедливая стоимость на дату оценки по курсу ЦБ'] = reestr_mini['Справедливая стоимость на дату оценки по курсу ЦБ']
reestr_initial['Дата оценки'] = reestr_mini['Текущая дата, на которую рассчитывалась справедливая стоимость ПФИ']
reestr_initial["Расхождение % относительно номинала сделки"] = fv_delta_percent
reestr_initial["Расхождение abs, тыс. руб."] = fv_delta_abs

reestr_initial.to_excel('reestr_fianl.xlsx')

reestr_initial.head()

Next step - адаптировать существующий файл под форварда/споты
